### 05: Кластеризация

In [ ]:
df = df_main_with_weather.copy()

df_unique = df.drop_duplicates(subset=['Широта WGS84', 'Долгота WGS84'])
df_unique = df_unique.dropna(subset=['Широта WGS84', 'Долгота WGS84'])
df_scaled = df_unique.copy()

#### Scatter plot координаты вышек

In [ ]:
scaler = StandardScaler()
df_scaled[['Широта_scaled', 'Долгота_scaled']] = scaler.fit_transform(df_unique[['Широта WGS84', 'Долгота WGS84']])


xy = np.vstack([df_scaled['Долгота_scaled'], df_scaled['Широта_scaled']])
z = stats.gaussian_kde(xy)(xy)

idx = z.argsort()
x_sorted = df_scaled['Долгота_scaled'].iloc[idx]
y_sorted = df_scaled['Широта_scaled'].iloc[idx]
z_sorted = z[idx]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(x_sorted, y_sorted, c=z_sorted, s=30, alpha=0.8,
                     cmap='hot', edgecolors='black', linewidth=0.3)
plt.colorbar(scatter, label='Плотность точек')
plt.xlabel('Долгота', fontsize=12)
plt.ylabel('Широта', fontsize=12)
plt.title('Scatter plot с градиентом плотности', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### K-means

In [ ]:
lat = df_unique['Широта WGS84'].values
lon = df_unique['Долгота WGS84'].values
lat_rad = np.radians(lat)
lon_rad = np.radians(lon)
lat0 = np.mean(lat_rad)
lon0 = np.mean(lon_rad)
R = 6371
x_km = R * (lon_rad - lon0) * np.cos(lat0)
y_km = R * (lat_rad - lat0)
X_km = np.column_stack((x_km, y_km))

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_scaled['KMeans_Cluster'] = kmeans.fit_predict(X_km)

kmeans_silhouette = silhouette_score(X_km, df_scaled['KMeans_Cluster'])
print(f"Silhouette Score финальной модели: {kmeans_silhouette:.4f}")


plt.figure(figsize=(10, 8))
plt.scatter(X_km[:, 0], X_km[:, 1], c=df_scaled['KMeans_Cluster'], cmap='tab10', s=30, alpha=0.6)

#### DBSCAN

In [ ]:
dbscan = DBSCAN(eps=4, min_samples=5)
labels = dbscan.fit_predict(X_km)

df_scaled['DBSCAN_Cluster'] = labels
mask = labels != -1
score = silhouette_score(X_km[mask], labels[mask])
print(f"Silhouette Score: {score:.4f}")

plt.figure(figsize=(10, 8))
plt.scatter(X_km[:, 0], X_km[:, 1], c=labels, cmap='tab10', s=30, alpha=0.6)

In [ ]:
from sklearn.cluster import KMeans, DBSCAN

def calculate_tower_metrics(y_true, y_pred, y_prob, tower_ids):
    metrics = []
    for tower_id in tower_ids.unique():
        mask = tower_ids == tower_id
        y_t = y_true[mask].values
        y_p = y_pred[mask]
        y_pr = y_prob[mask]

        if len(y_t) < 10:
            continue

        if len(np.unique(y_t)) == 2:
            auc = roc_auc_score(y_t, y_pr)
            pr_auc = average_precision_score(y_t, y_pr)
        else:
            auc = np.nan
            pr_auc = np.nan

        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            'Master_Site': tower_id,
            'auc_roc': auc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })
    return pd.DataFrame(metrics)

tower_metrics = calculate_tower_metrics(y_test, y_pred_lgb, y_prob_lgb, test_df['Master Site'])

feature_cols = ['auc_roc', 'precision', 'recall', 'f1_score']
X_cluster = tower_metrics[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("K-MEANS КЛАСТЕРИЗАЦИЯ")
print("="*60)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
tower_metrics.loc[X_cluster.index, 'kmeans_cluster'] = kmeans.fit_predict(X_scaled)

for cluster_id in range(n_clusters):
    cluster_data = tower_metrics[tower_metrics['kmeans_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")
    print(f"  Recall:   {cluster_data['recall'].mean():.3f}")

print("\n" + "="*60)
print("DBSCAN КЛАСТЕРИЗАЦИЯ")
print("="*60)

dbscan = DBSCAN(eps=0.8, min_samples=3)
tower_metrics.loc[X_cluster.index, 'dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_dbscan_clusters = len(set(tower_metrics['dbscan_cluster'])) - (1 if -1 in tower_metrics['dbscan_cluster'].values else 0)
n_noise = list(tower_metrics['dbscan_cluster']).count(-1)

print(f"Найдено кластеров: {n_dbscan_clusters}")
print(f"Шумовых точек: {n_noise}")

for cluster_id in sorted(set(tower_metrics['dbscan_cluster'])):
    if cluster_id == -1:
        continue
    cluster_data = tower_metrics[tower_metrics['dbscan_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")



In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

def calculate_tower_metrics(y_true, y_pred, y_prob, tower_ids):
    metrics = []
    for tower_id in tower_ids.unique():
        mask = tower_ids == tower_id
        y_t = y_true[mask].values
        y_p = y_pred[mask]
        y_pr = y_prob[mask]

        if len(y_t) < 10:
            continue

        if len(np.unique(y_t)) == 2:
            auc = roc_auc_score(y_t, y_pr)
            pr_auc = average_precision_score(y_t, y_pr)
        else:
            auc = np.nan
            pr_auc = np.nan

        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            'Master_Site': tower_id,
            'auc_roc': auc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })
    return pd.DataFrame(metrics)

tower_metrics = calculate_tower_metrics(y_test, y_pred_lgb, y_prob_lgb, test_df['Master Site'])

# 2. Готовим данные для кластеризации
feature_cols = ['auc_roc', 'precision', 'recall', 'f1_score']
X_cluster = tower_metrics[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# 3. K-Means
print("K-MEANS КЛАСТЕРИЗАЦИЯ")
print("="*60)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
tower_metrics.loc[X_cluster.index, 'kmeans_cluster'] = kmeans.fit_predict(X_scaled)

for cluster_id in range(n_clusters):
    cluster_data = tower_metrics[tower_metrics['kmeans_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")
    print(f"  Recall:   {cluster_data['recall'].mean():.3f}")

# 4. DBSCAN
print("\n" + "="*60)
print("DBSCAN КЛАСТЕРИЗАЦИЯ")
print("="*60)

dbscan = DBSCAN(eps=0.8, min_samples=3)
tower_metrics.loc[X_cluster.index, 'dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_dbscan_clusters = len(set(tower_metrics['dbscan_cluster'])) - (1 if -1 in tower_metrics['dbscan_cluster'].values else 0)
n_noise = list(tower_metrics['dbscan_cluster']).count(-1)

print(f"Найдено кластеров: {n_dbscan_clusters}")
print(f"Шумовых точек: {n_noise}")

for cluster_id in sorted(set(tower_metrics['dbscan_cluster'])):
    if cluster_id == -1:
        continue
    cluster_data = tower_metrics[tower_metrics['dbscan_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")

# 5. Сравнение с географической кластеризацией
print("\n" + "="*60)
print("СРАВНЕНИЕ С ГЕОГРАФИЧЕСКОЙ КЛАСТЕРИЗАЦИЕЙ")
print("="*60)

geo_col = 'Subregion'  

if geo_col in test_df.columns:
    tower_geo = test_df[['Master Site', geo_col]].drop_duplicates()
    merged = tower_metrics.merge(tower_geo, on='Master_Site', how='left')

    print("\nK-Means × География:")
    print(pd.crosstab(merged['kmeans_cluster'], merged[geo_col]))

    print("\nDBSCAN × География:")
    print(pd.crosstab(merged['dbscan_cluster'], merged[geo_col]))
else:
    print(f"Колонка {geo_col} не найдена!")

# Сохранение
tower_metrics.to_csv('tower_clusters.csv', index=False)
print("\nРезультаты сохранены в tower_clusters.csv")